# Prediction Model - EDA Workbook

This notebook organizes EDA for **DB for DTM Project.xlsx** using:
- Markdown explanations,
- code cells for each step,
- visible intermediate outputs.


## 1) Setup

Import libraries and set display options.


In [59]:
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 2) Locate Excel File and Inspect Sheets

Check file location and available sheet names.


In [60]:
# Notebook folder: .../vem-product/Classification pattern
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR / "DB for DTM Project.xlsx",
    NOTEBOOK_DIR.parent / "DB for DTM Project.xlsx",
]

FILE_PATH = next((p for p in CANDIDATES if p.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError(
        "DB for DTM Project.xlsx not found. Put it in current folder or parent folder."
    )

FILE_PATH


WindowsPath('c:/Users/sveta/Documents/vem-product/DB for DTM Project.xlsx')

In [61]:
xl = pd.ExcelFile(FILE_PATH)

sheet_overview = pd.DataFrame({
    "sheet_name": xl.sheet_names,
    "rows": [pd.read_excel(FILE_PATH, sheet_name=s).shape[0] for s in xl.sheet_names],
    "cols": [pd.read_excel(FILE_PATH, sheet_name=s).shape[1] for s in xl.sheet_names],
})

sheet_overview


,sheet_name,rows,cols
0,LOB,345,2
1,Codice Contabile Articoli,57,3
2,Associazioni Cod. Art. - LOB,22655,5


## 3) Load Main Sheet

Main sheet for classification: **Associazioni Cod. Art. - LOB**.


In [62]:
SHEET_NAME = "Associazioni Cod. Art. - LOB"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = [str(c).strip() for c in df.columns]

print(f"shape: {df.shape}")
print(f"columns: {list(df.columns)}")

# preview
head_rows = df.head(10)
head_rows


shape: (22655, 5)
columns: ['Codice Articolo', 'Lob Associata', 'Nome LOB', 'Inventario', 'Descrizione Articolo']


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...
3,00054569,3004,CLOUD VARIE HW,Inventario,adattatore vga-hdmi
4,000ERIDT29,4002,ALTRO MATERIALE,Inventario,ERICSSON DT290/292 BATTERIA CORDLESS
5,001DELTA-E,24001,BUILDING AUTOMATION,Inventario,FOTOCELLULA CAME DA ESTERNO COPPIA RICEVITORE/...
6,0021X145,3004,CLOUD VARIE HW,Inventario,CUSTODIA MORBIDA DCC-80
7,002349147,3004,CLOUD VARIE HW,Inventario,E350 E352DN 3 ANNI TOTALI (1+2)
8,002R3G6,4002,ALTRO MATERIALE,Inventario,CELLULARE NOKIA E52 NAVY
9,00413.CK.B,1002,CABLAGGIO ALTERNATIVO,Inventario,Presa mult. 3P BIANCA


## 4) Helper Functions

Utility functions for column detection and keyword splitting.


In [63]:
def normalize_colname(name: str) -> str:
    return re.sub(r"\s+", " ", str(name).strip().lower())


def find_column(columns, patterns):
    normalized = {c: normalize_colname(c) for c in columns}
    for pattern in patterns:
        regex = re.compile(pattern, flags=re.IGNORECASE)
        for col, norm in normalized.items():
            if regex.search(norm):
                return col
    return None


def split_keywords(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[;,/|]+", text)
    return [p.strip().lower() for p in parts if p.strip()]


## 5) Column Roles (Input / Target)

Identify target and input columns.


In [64]:
target_col = find_column(df.columns, [r"\binventario\b", r"\btarget\b", r"\blabel\b", r"\bclass\b"])
lob_col = find_column(df.columns, [r"\blob associata\b", r"\blob\b"])
nome_lob_col = find_column(df.columns, [r"\bnome\s*lob\b", r"\blob name\b"])
code_col = find_column(df.columns, [r"\bcod(ic|e|ice)?\b", r"\bcode\b", r"\bitem code\b"])
keywords_col = find_column(df.columns, [r"\bdescr(izione)?\b", r"\bkeywords?\b", r"\bkeyword\b"])

roles_df = pd.DataFrame({"column": df.columns})
roles_df["role"] = roles_df["column"].apply(lambda c: "target" if c == target_col else "input")

print({
    "target_col": target_col,
    "lob_col": lob_col,
    "nome_lob_col": nome_lob_col,
    "code_col": code_col,
    "keywords_col": keywords_col,
})

roles_df


{'target_col': 'Inventario', 'lob_col': 'Lob Associata', 'nome_lob_col': 'Nome LOB', 'code_col': 'Codice Articolo', 'keywords_col': 'Descrizione Articolo'}


,column,role
0,Codice Articolo,input
1,Lob Associata,input
2,Nome LOB,input
3,Inventario,target
4,Descrizione Articolo,input


## 6) Basic Data Profile

Overview of dtypes, uniqueness, and missing values.


In [65]:
profile_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes.values],
    "non_null": df.notna().sum().values,
    "null": df.isna().sum().values,
    "null_pct": (df.isna().mean().values * 100).round(3),
    "n_unique": df.nunique(dropna=True).values,
})

profile_df.sort_values("null", ascending=False)


,column,dtype,non_null,null,null_pct,n_unique
4,Descrizione Articolo,object,22595,60,0.265,21283
0,Codice Articolo,object,22655,0,0.000,22655
1,Lob Associata,int64,22655,0,0.000,117
2,Nome LOB,object,22655,0,0.000,115
3,Inventario,object,22655,0,0.000,3


## 7) Missing Values Check


In [66]:
missing_df = (
    pd.DataFrame({"column": df.columns, "missing_count": df.isna().sum().values})
    .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(3))
    .sort_values("missing_count", ascending=False)
)

missing_df


,column,missing_count,missing_pct
4,Descrizione Articolo,60,0.265
0,Codice Articolo,0,0.000
1,Lob Associata,0,0.000
2,Nome LOB,0,0.000
3,Inventario,0,0.000


## 8) Duplicate Rows Check


In [67]:
duplicate_mask = df.duplicated(keep=False)
duplicates_df = df[duplicate_mask].copy()

print(f"duplicate rows (all columns): {duplicate_mask.sum()}")

duplicates_df.head(20)


duplicate rows (all columns): 0


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo


## 9) Target Conflict Check

Conflict = same inputs mapped to different targets.


In [68]:
conflicts_df = pd.DataFrame()

if target_col is None:
    print("Target column not found, skip conflict check.")
else:
    input_cols = [c for c in df.columns if c != target_col]
    grouped = (
        df.groupby(input_cols, dropna=False)[target_col]
        .nunique(dropna=True)
        .reset_index(name="target_nunique")
    )
    key_conflicts = grouped[grouped["target_nunique"] > 1][input_cols]

    if not key_conflicts.empty:
        conflicts_df = key_conflicts.merge(df, on=input_cols, how="left")

    print(f"conflict groups: {len(key_conflicts)}")
    print(f"conflict rows: {len(conflicts_df)}")

conflicts_df.head(20)


conflict groups: 0
conflict rows: 0


""


## 10) Nome LOB vs Lob Associata (1:1 Check)

Verify whether `Nome LOB` is a strict textual expansion of `Lob Associata`.


In [69]:
if lob_col is None or nome_lob_col is None:
    print("LOB/Nome LOB columns not found.")
else:
    pairs = df[[lob_col, nome_lob_col]].dropna().drop_duplicates()

    lob_to_nome = (
        pairs.groupby(lob_col)[nome_lob_col]
        .nunique()
        .reset_index(name="nome_per_lob")
        .sort_values("nome_per_lob", ascending=False)
    )

    nome_to_lob = (
        pairs.groupby(nome_lob_col)[lob_col]
        .nunique()
        .reset_index(name="lob_per_nome")
        .sort_values("lob_per_nome", ascending=False)
    )

    print("LOB -> many Nome LOB:", int((lob_to_nome["nome_per_lob"] > 1).sum()))
    print("Nome LOB -> many LOB:", int((nome_to_lob["lob_per_nome"] > 1).sum()))

    print("Problematic Nome LOB (mapped to >1 LOB):")
    display(nome_to_lob[nome_to_lob["lob_per_nome"] > 1])

    display(
        pairs[pairs[nome_lob_col].isin(
            nome_to_lob.loc[nome_to_lob["lob_per_nome"] > 1, nome_lob_col]
        )]
        .sort_values([nome_lob_col, lob_col])
    )


LOB -> many Nome LOB: 0
Nome LOB -> many LOB: 2
Problematic Nome LOB (mapped to >1 LOB):


,Nome LOB,lob_per_nome
114,VIDEOSORVEGLIANZA,2
15,BUILDING AUTOMATION,2


,Lob Associata,Nome LOB
2171,1011,BUILDING AUTOMATION
5,24001,BUILDING AUTOMATION
57,1015,VIDEOSORVEGLIANZA
20,2013,VIDEOSORVEGLIANZA


## 11) Requested Simple Checks (Raw)

This section contains the exact simple checks requested:
- verify whether `Nome LOB` is just a decoding of `Lob Associata`,
- simple frequency table `code prefix vs LOB`,
- simple frequency table `keywords vs Inventario`.


In [70]:
if lob_col is None or nome_lob_col is None:
    print("LOB/Nome LOB columns not found")
else:
    pairs_simple = df[[lob_col, nome_lob_col]].dropna().drop_duplicates()
    lob_to_nome_simple = (
        pairs_simple.groupby(lob_col)[nome_lob_col]
        .nunique()
        .reset_index(name="nome_per_lob")
        .sort_values("nome_per_lob", ascending=False)
    )
    nome_to_lob_simple = (
        pairs_simple.groupby(nome_lob_col)[lob_col]
        .nunique()
        .reset_index(name="lob_per_nome")
        .sort_values("lob_per_nome", ascending=False)
    )

    lob_to_many_nome_simple = lob_to_nome_simple[lob_to_nome_simple["nome_per_lob"] > 1]
    nome_to_many_lob_simple = nome_to_lob_simple[nome_to_lob_simple["lob_per_nome"] > 1]

    is_nome_lob_simple_decoding = (
        lob_to_many_nome_simple.empty and nome_to_many_lob_simple.empty
    )

    print("Is Nome LOB just a strict 1:1 decoding of Lob Associata?", is_nome_lob_simple_decoding)
    print("LOB -> many Nome LOB:", len(lob_to_many_nome_simple))
    print("Nome LOB -> many LOB:", len(nome_to_many_lob_simple))

    if not nome_to_many_lob_simple.empty:
        print("\nProblematic Nome LOB values (>1 LOB):")
        display(nome_to_many_lob_simple)

        display(
            pairs_simple[pairs_simple[nome_lob_col].isin(nome_to_many_lob_simple[nome_lob_col])]
            .sort_values([nome_lob_col, lob_col])
        )


Is Nome LOB just a strict 1:1 decoding of Lob Associata? False
LOB -> many Nome LOB: 0
Nome LOB -> many LOB: 2

Problematic Nome LOB values (>1 LOB):


,Nome LOB,lob_per_nome
114,VIDEOSORVEGLIANZA,2
15,BUILDING AUTOMATION,2


,Lob Associata,Nome LOB
2171,1011,BUILDING AUTOMATION
5,24001,BUILDING AUTOMATION
57,1015,VIDEOSORVEGLIANZA
20,2013,VIDEOSORVEGLIANZA


In [71]:
if code_col is None or lob_col is None:
    print("code_col/lob_col not found")
    freq_prefix_vs_lob = pd.DataFrame()
else:
    temp_prefix = df[[code_col, lob_col]].dropna().copy()
    code_s = temp_prefix[code_col].astype(str).str.strip().str.upper()
    alpha_prefix = code_s.str.extract(r"^([A-Z]+)", expand=False)
    num_prefix2 = code_s.str.extract(r"^(\d{2})", expand=False)
    temp_prefix["code_prefix"] = alpha_prefix.fillna(num_prefix2).fillna("OTHER")

    freq_prefix_vs_lob = (
        temp_prefix.groupby(["code_prefix", lob_col])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    display(freq_prefix_vs_lob.head(30))


,code_prefix,Lob Associata,count
1324,C,2002,428
3869,PAN,20001,368
929,AIR,2001,243
5571,X,3014,236
3844,P,3010,222
5131,UCS,3009,202
3145,LIC,2016,200
2328,FC,6007,199
5527,WS,2002,195
1645,CP,4001,173


In [72]:
if keywords_col is None or target_col is None:
    print("keywords_col/target_col not found")
    freq_keywords_vs_inventario = pd.DataFrame()
else:
    temp_kw_simple = df[[keywords_col, target_col]].dropna().copy()
    temp_kw_simple["keyword"] = temp_kw_simple[keywords_col].apply(split_keywords)
    temp_kw_simple = temp_kw_simple.explode("keyword")
    temp_kw_simple = temp_kw_simple[temp_kw_simple["keyword"].notna() & (temp_kw_simple["keyword"] != "")]

    freq_keywords_vs_inventario = (
        temp_kw_simple.groupby(["keyword", target_col])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    display(freq_keywords_vs_inventario.head(30))


,keyword,Inventario,count
172,-c,Inventario,291
18893,normal,Inventario,154
22285,s,Inventario,117
12841,factory integrated,Inventario,106
24484,sw,Inventario,103
627,100,Inventario,91
21871,renew,Inventario,80
20256,per-0.1tb,Inventario,72
7456,bretella di permutazione cat. 6,Inventario,71
7967,cable,Inventario,68


## 12) Data Preparation Block - LOB Label Cleaning

This block documents target-label cleaning for grouping LOB codes.

Priority of rules:
1. Name-based reconciliation with prefix coupling (`Nome LOB` + first 2 digits of `current_lob` against LOB sheet)
2. Family-based unique fallback
3. Description similarity fallback
4. Manual review / no evidence



In [73]:
DATA_QUALITY_DIR = NOTEBOOK_DIR / 'data_quality'

candidates_path = DATA_QUALITY_DIR / 'grouping_lob_candidates.csv'
changed_path = DATA_QUALITY_DIR / 'grouping_lob_changed_rows_preview.csv'
auto_path = DATA_QUALITY_DIR / 'grouping_lob_autofix_safe.csv'
manual_path = DATA_QUALITY_DIR / 'grouping_lob_manual_review.csv'
cleaned_path = DATA_QUALITY_DIR / 'associazioni_with_lob_cleaned.csv'
summary_txt_path = DATA_QUALITY_DIR / 'grouping_lob_summary.txt'

required_paths = [candidates_path, changed_path, auto_path, manual_path, cleaned_path, summary_txt_path]
missing = [str(x) for x in required_paths if not x.exists()]
if missing:
    raise FileNotFoundError('Missing data_quality files: ' + ', '.join(missing))

cand_df = pd.read_csv(candidates_path)
display(cand_df['decision'].value_counts(dropna=False).rename_axis('decision').reset_index(name='count'))

fixed_decisions = ['fix_by_name_prefix2', 'fix_by_family_unique', 'fix_by_description']
remaining_decisions = ['manual_review', 'no_evidence']

fixed_count = int(cand_df[cand_df['decision'].isin(fixed_decisions)].shape[0])
remaining_count = int(cand_df[cand_df['decision'].isin(remaining_decisions)].shape[0])

print('Data Preparation (LOB cleaning)')
print('Fixed rows:', fixed_count)
print('Remaining manual/no-evidence rows:', remaining_count)
print('Summary file:')
print(summary_txt_path.read_text(encoding='utf-8'))



,decision,count
0,fix_by_name_prefix2,54


Data Preparation (LOB cleaning)
Fixed rows: 54
Remaining manual/no-evidence rows: 0
Summary file:
grouping_rows_total=54
fixed_total=54
remaining_total=0
fixed_by_name_prefix2=54
fixed_by_family_unique=0
fixed_by_description=0
manual_review=0
no_evidence=0


In [74]:
changed_df = pd.read_csv(changed_path)
auto_fix_df = pd.read_csv(auto_path)
manual_df = pd.read_csv(manual_path)
cleaned_df = pd.read_csv(cleaned_path)

print('Changed rows preview:')
display(changed_df.head(30))

print('Auto-fix rows:')
display(auto_fix_df.head(30))

print('Manual review rows:')
display(manual_df.head(30))

print('Cleaning decision distribution in full dataset:')
col = 'LOB Cleaning Decision' if 'LOB Cleaning Decision' in cleaned_df.columns else None
if col:
    display(cleaned_df[col].value_counts(dropna=False).rename_axis('decision').reset_index(name='count'))



Changed rows preview:


,Codice Articolo,Lob Associata,Lob Associata Orig Norm,Lob Associata Cleaned,Nome LOB,LOB Cleaning Decision
0,0151254-24,1,1,1,IMPIANTI,fix_by_name_prefix2
1,14208-08,4,4,4999,COLLABORATION,fix_by_name_prefix2
2,16QT2200315,17,17,17999,DATACENTER INFRASTRUCTURE,fix_by_name_prefix2
3,33-912.0931,4,4,4999,COLLABORATION,fix_by_name_prefix2
4,33-917.227,4,4,4999,COLLABORATION,fix_by_name_prefix2
5,33-918.002,4,4,4999,COLLABORATION,fix_by_name_prefix2
6,798-0714,17,17,17999,DATACENTER INFRASTRUCTURE,fix_by_name_prefix2
7,81681,4,4,4999,COLLABORATION,fix_by_name_prefix2
8,ABBIGLIAMENTO,99,99,99999,VARIE,fix_by_name_prefix2
9,ACTELKIT,4,4,4999,COLLABORATION,fix_by_name_prefix2


Auto-fix rows:


,Codice Articolo,current_lob,current_nome_lob,family,name_candidates_all,name_candidates_prefix2,suggested_lob_name,suggested_lob_family,family_lob_nunique,top_match_code,top_match_lob,top_match_nome_lob,top_similarity,second_similarity,similarity_gap,decision,suggested_lob,rule_priority
0,0151254-24,1,IMPIANTI,0151254,01,01,1,1002.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,1,1
1,14208-08,4,COLLABORATION,14208,04|04999,04|04999,4999,3004.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1
2,16QT2200315,17,DATACENTER INFRASTRUCTURE,16QT2200315,17|17999,17|17999,17999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,17999,1
3,33-912.0931,4,COLLABORATION,33,04|04999,04|04999,4999,4009.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1
4,33-917.227,4,COLLABORATION,33,04|04999,04|04999,4999,4009.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1
5,33-918.002,4,COLLABORATION,33,04|04999,04|04999,4999,4009.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1
6,798-0714,17,DATACENTER INFRASTRUCTURE,798,17|17999,17|17999,17999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,17999,1
7,81681,4,COLLABORATION,81681,04|04999,04|04999,4999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1
8,ABBIGLIAMENTO,99,VARIE,ABBIGLIAMENTO,99|99999,99|99999,99999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,99999,1
9,ACTELKIT,4,COLLABORATION,ACTELKIT,04|04999,04|04999,4999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fix_by_name_prefix2,4999,1


Manual review rows:


,Codice Articolo,current_lob,current_nome_lob,family,name_candidates_all,name_candidates_prefix2,suggested_lob_name,suggested_lob_family,family_lob_nunique,top_match_code,top_match_lob,top_match_nome_lob,top_similarity,second_similarity,similarity_gap,decision,suggested_lob,rule_priority


Cleaning decision distribution in full dataset:


,decision,count
0,unchanged,22601
1,fix_by_name_prefix2,54


## 13) Prefix Signal Analysis (Code -> LOB)

Instead of raw frequencies, this section finds prefixes with strong LOB signal using:
- `support` = number of rows for prefix,
- `confidence` = share of dominant LOB within prefix,
- `lift` = confidence / global share of that LOB.


In [75]:
if code_col is None or lob_col is None:
    print("code_col/lob_col not found")
    code_lob_freq = pd.DataFrame()
    prefix_rules = pd.DataFrame()
else:
    temp = df[[code_col, lob_col]].dropna().copy()
    code_s = temp[code_col].astype(str).str.strip().str.upper()

    # Prefix strategy: alpha prefix first, otherwise first 2 digits.
    alpha_prefix = code_s.str.extract(r"^([A-Z]+)", expand=False)
    num_prefix2 = code_s.str.extract(r"^(\d{2})", expand=False)
    temp["code_prefix"] = alpha_prefix.fillna(num_prefix2).fillna("OTHER")

    code_lob_freq = (
        temp.groupby(["code_prefix", lob_col]).size().reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    global_lob_share = temp[lob_col].value_counts(normalize=True).rename("global_share")

    grp = code_lob_freq.copy()
    grp["prefix_total"] = grp.groupby("code_prefix")["count"].transform("sum")
    grp["confidence"] = grp["count"] / grp["prefix_total"]
    grp = grp.merge(global_lob_share, left_on=lob_col, right_index=True, how="left")
    grp["lift"] = grp["confidence"] / grp["global_share"]

    prefix_rules = (
        grp.sort_values(["code_prefix", "confidence", "count"], ascending=[True, False, False])
        .drop_duplicates(subset=["code_prefix"])
        .rename(columns={lob_col: "dominant_lob", "count": "dominant_count"})
    )

    # Keep only prefixes with enough evidence.
    prefix_rules = prefix_rules[prefix_rules["prefix_total"] >= 20].copy()
    prefix_rules = prefix_rules.sort_values(["confidence", "lift", "prefix_total"], ascending=False)

    print("Top prefix->LOB rules (support>=20):")
    display(prefix_rules[["code_prefix", "dominant_lob", "prefix_total", "dominant_count", "confidence", "lift"]].head(30))

    print("Raw top combinations (prefix, LOB):")
    display(code_lob_freq.head(20))


Top prefix->LOB rules (support>=20):


,code_prefix,dominant_lob,prefix_total,dominant_count,confidence,lift
1939,DG,3002,27,27,1.000000,150.033113
5212,UTP,1008,48,48,1.000000,54.198565
2495,FZ,1008,25,25,1.000000,54.198565
3647,NPC,1001,31,31,1.000000,45.491968
1663,CPC,1001,25,25,1.000000,45.491968
1660,CPAP,6002,89,89,1.000000,38.463497
1662,CPAPSG,6002,25,25,1.000000,38.463497
2314,FAS,3014,66,66,1.000000,27.729498
848,ACDC,17001,33,33,1.000000,22.475198
835,ACAC,17001,22,22,1.000000,22.475198


Raw top combinations (prefix, LOB):


,code_prefix,Lob Associata,count
1324,C,2002,428
3869,PAN,20001,368
929,AIR,2001,243
5571,X,3014,236
3844,P,3010,222
5131,UCS,3009,202
3145,LIC,2016,200
2328,FC,6007,199
5527,WS,2002,195
1645,CP,4001,173


## 14) Keyword Signal Analysis (Description -> Inventario)

Raw keyword counts are often noisy because the dataset is imbalanced.
This section computes keyword-level signal with:
- `support` (keyword frequency),
- `confidence` = P(target | keyword),
- `lift` = confidence / P(target).


In [76]:
if keywords_col is None or target_col is None:
    print("keywords_col/target_col not found")
    kw_target_freq = pd.DataFrame()
    keyword_rules = pd.DataFrame()
else:
    stopwords = {
        "di", "da", "per", "con", "del", "della", "delle", "degli",
        "the", "and", "for", "with", "non", "una", "uno", "all",
    }

    def tokenize_desc(v):
        if pd.isna(v):
            return []
        txt = str(v).lower()
        txt = re.sub(r"[^a-z0-9]+", " ", txt)
        toks = [t for t in txt.split() if len(t) >= 3 and t not in stopwords]
        return toks

    temp = df[[keywords_col, target_col]].dropna().copy()
    temp["keyword"] = temp[keywords_col].apply(tokenize_desc)
    temp = temp.explode("keyword")
    temp = temp[temp["keyword"].notna() & (temp["keyword"] != "")]

    kw_target_freq = (
        temp.groupby(["keyword", target_col]).size().reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    kw_total = temp.groupby("keyword").size().rename("support")
    base_rate = temp[target_col].value_counts(normalize=True).rename("base_rate")

    kw_stats = kw_target_freq.merge(kw_total, on="keyword", how="left")
    kw_stats["confidence"] = kw_stats["count"] / kw_stats["support"]
    kw_stats = kw_stats.merge(base_rate, left_on=target_col, right_index=True, how="left")
    kw_stats["lift"] = kw_stats["confidence"] / kw_stats["base_rate"]

    # Candidate rules with enough support.
    keyword_rules = kw_stats[kw_stats["support"] >= 15].copy()
    keyword_rules = keyword_rules.sort_values(["lift", "support", "confidence"], ascending=False)

    print("Top keyword->target signals (support>=15):")
    display(keyword_rules[["keyword", target_col, "support", "count", "confidence", "lift"]].head(40))

    print("Top keywords per target (by lift, support>=15):")
    for cls in keyword_rules[target_col].dropna().unique():
        print(f"\nTarget = {cls}")
        display(
            keyword_rules[keyword_rules[target_col] == cls][["keyword", "support", "count", "confidence", "lift"]]
            .head(15)
        )


Top keyword->target signals (support>=15):


,keyword,Inventario,support,count,confidence,lift
5756,parts,Assistenza,24,3,0.125000,276.025000
8334,4hr,Assistenza,22,2,0.090909,200.745455
15979,advisor,Assistenza,15,1,0.066667,147.213333
6707,mths,Assistenza,33,2,0.060606,133.830303
4948,site,Assistenza,57,3,0.052632,116.221053
12173,supportedge,Assistenza,21,1,0.047619,105.152381
15722,9x5,Assistenza,24,1,0.041667,92.008333
15776,netbotz,Assistenza,25,1,0.040000,88.328000
13067,2021,Assistenza,26,1,0.038462,84.930769
5580,ext,Assistenza,81,3,0.037037,81.785185


Top keywords per target (by lift, support>=15):

Target = Assistenza


,keyword,support,count,confidence,lift
5756,parts,24,3,0.125000,276.025000
8334,4hr,22,2,0.090909,200.745455
15979,advisor,15,1,0.066667,147.213333
6707,mths,33,2,0.060606,133.830303
4948,site,57,3,0.052632,116.221053
12173,supportedge,21,1,0.047619,105.152381
15722,9x5,24,1,0.041667,92.008333
15776,netbotz,25,1,0.040000,88.328000
13067,2021,26,1,0.038462,84.930769
5580,ext,81,3,0.037037,81.785185



Target = Non in inventario


,keyword,support,count,confidence,lift
24,panoptikon,388,388,1.0,7.266206
173,soar,109,109,1.0,7.266206
495,addr,47,47,1.0,7.266206
530,response,43,43,1.0,7.266206
702,1499,34,34,1.0,7.266206
708,1999,34,34,1.0,7.266206
732,tactical,32,32,1.0,7.266206
761,349,31,31,1.0,7.266206
787,749,30,30,1.0,7.266206
819,149,29,29,1.0,7.266206



Target = Inventario


,keyword,support,count,confidence,lift
4,port,634,634,1.0,1.160195
5,cable,599,599,1.0,1.160195
9,usb,526,526,1.0,1.160195
21,lszh,398,398,1.0,1.160195
22,bretella,397,397,1.0,1.160195
25,hdmi,387,387,1.0,1.160195
34,cord,334,334,1.0,1.160195
44,patch,293,293,1.0,1.160195
50,duplex,272,272,1.0,1.160195
58,mount,222,222,1.0,1.160195


## 15) Quick Summary (for report)

Compact EDA summary.


In [77]:
summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "target_col": target_col,
    "lob_col": lob_col,
    "nome_lob_col": nome_lob_col,
    "code_col": code_col,
    "keywords_col": keywords_col,
    "duplicate_rows": int(duplicate_mask.sum()),
    "columns_with_missing": int((missing_df['missing_count'] > 0).sum()),
    "target_conflict_rows": int(len(conflicts_df)) if target_col else None,
}

if lob_col and nome_lob_col:
    summary["lob_to_many_nome"] = int((lob_to_nome["nome_per_lob"] > 1).sum())
    summary["nome_to_many_lob"] = int((nome_to_lob["lob_per_nome"] > 1).sum())

pd.Series(summary)


rows                                   22655
columns                                    5
target_col                        Inventario
lob_col                        Lob Associata
nome_lob_col                        Nome LOB
code_col                     Codice Articolo
keywords_col            Descrizione Articolo
duplicate_rows                             0
columns_with_missing                       1
target_conflict_rows                       0
lob_to_many_nome                           0
nome_to_many_lob                           2
dtype: object

## 16) Optional Export of Intermediate Results

Run the next cell to save intermediate CSV outputs.


In [78]:
EXPORT_DIR = NOTEBOOK_DIR / "eda_outputs_notebook"
EXPORT_DIR.mkdir(exist_ok=True)

profile_df.to_csv(EXPORT_DIR / "columns_overview.csv", index=False)
roles_df.to_csv(EXPORT_DIR / "column_roles.csv", index=False)
missing_df.to_csv(EXPORT_DIR / "missing_values.csv", index=False)
duplicates_df.to_csv(EXPORT_DIR / "duplicate_rows.csv", index=False)
conflicts_df.to_csv(EXPORT_DIR / "target_conflicts.csv", index=False)

if 'code_lob_freq' in globals() and not code_lob_freq.empty:
    code_lob_freq.to_csv(EXPORT_DIR / "freq_code_prefix_vs_lob.csv", index=False)

if 'kw_target_freq' in globals() and not kw_target_freq.empty:
    kw_target_freq.to_csv(EXPORT_DIR / "freq_keywords_vs_target.csv", index=False)

if lob_col and nome_lob_col:
    pairs.to_csv(EXPORT_DIR / "lob_nome_pairs.csv", index=False)
    lob_to_nome.to_csv(EXPORT_DIR / "lob_to_nome_cardinality.csv", index=False)
    nome_to_lob.to_csv(EXPORT_DIR / "nome_to_lob_cardinality.csv", index=False)

print("Saved to:", EXPORT_DIR)

if 'freq_prefix_vs_lob' in globals() and not freq_prefix_vs_lob.empty:
    freq_prefix_vs_lob.to_csv(EXPORT_DIR / "freq_prefix_vs_lob_simple.csv", index=False)

if 'freq_keywords_vs_inventario' in globals() and not freq_keywords_vs_inventario.empty:
    freq_keywords_vs_inventario.to_csv(EXPORT_DIR / "freq_keywords_vs_inventario_simple.csv", index=False)

if 'nome_to_lob_simple' in globals():
    nome_to_lob_simple.to_csv(EXPORT_DIR / "nome_to_lob_cardinality_simple.csv", index=False)


Saved to: c:\Users\sveta\Documents\vem-product\Classification pattern\eda_outputs_notebook
